# ChiralFold Results Dashboard

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/ChiralFold_Results_Dashboard.ipynb)

> **⚠️ Auto-setup warning:** Clicking **Open in Colab** clones the ChiralFold repository and installs dependencies when you run the first code cells. Expect **~1–2 minutes** on a fresh Colab runtime. **Do not skip the install cell** — visualizations read from bundled `results/` artefacts.

Interactive dashboard for all ChiralFold benchmark results: D-residue PDB survey, experimental validation, Ramachandran agreement with wwPDB, AF3 chirality correction, mmCIF re-verification, and Colab demo highlights.

**Paper headline Ramachandran:** n=362 (5M2K excluded), Spearman ρ=0.52. **Colab replications** at n=155 and n=285 are independent runs — see interpretation panel below.


In [ ]:
# Cell 1 — Install ChiralFold + Plotly (~1–2 min on fresh Colab)
!pip -q install "chiralfold==3.5.1" plotly pandas matplotlib


In [ ]:
# Cell 2 — Clone repo if results/ is not already present
import os, subprocess
if not os.path.isdir('results'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Tommaso-R-Marena/ChiralFold.git', 'ChiralFold'], check=True)
    os.chdir('ChiralFold')
print('Working directory:', os.getcwd())
print('Manifest exists:', os.path.isfile('results/colab_integrated_manifest.json'))


In [ ]:
# Cell 3 — Load integrated manifest and key artefacts
import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path('.')
with open(ROOT / 'results/colab_integrated_manifest.json') as f:
    manifest = json.load(f)

def load_json(rel):
    with open(ROOT / rel) as f:
        return json.load(f)

d_survey = load_json('results/d_residue_verification_summary.json')
exp_val = load_json('results/experimental_validation_report.json')
af3 = load_json('results/af3_resource_benchmark.json')
chainfix = load_json('results/ramachandran_279struct_chainfix_summary.json')
error_cls = load_json('results/error_classification.json')
mmcif = load_json('results/mmcif_d_residue_expansion_summary.json')

print('ChiralFold Results Dashboard — loaded', len(manifest['runs']), 'benchmark runs')
print('Package version in manifest:', manifest.get('chiralfold_version'))
print(manifest['interpretation']['paper_headline_ramachandran'])


In [ ]:
print(f"Legacy D-residue survey: {d['checkable_residues']:,} residues · {d['l_error']} errors")
known = mmcif.get('known_error_cohort', {})
uni = mmcif.get('universe_cohort', {})
print(f"mmCIF known-error re-verify: {known.get('n_errors', mmcif.get('n_errors'))} errors "
      f"across {known.get('n_structures', 16)} structures")
print(f"mmCIF-only universe: {uni.get('n_structures_requested', '?')} structures · "
      f"{uni.get('n_errors', 0)} new errors {uni.get('error_pdbs', [])}")


In [ ]:
# Cell 5 — Ramachandran sample-size progression (Spearman ρ)
prog = pd.DataFrame(manifest['ramachandran_progression'])
colors = {'repo_manuscript': '#2563eb', 'colab': '#059669', 'repo': '#6b7280', 'repo_historical': '#d1d5db'}

fig = go.Figure()
for src, grp in prog.groupby('source'):
    fig.add_trace(go.Scatter(
        x=grp['n'], y=grp['spearman_rho'], mode='markers+text',
        name=src, text=grp['label'], textposition='top center',
        marker=dict(size=14, color=colors.get(src, '#333')),
    ))
fig.update_layout(
    title='Ramachandran agreement with wwPDB: Spearman ρ vs sample size',
    xaxis_title='Structures analyzed (n)',
    yaxis_title='Spearman ρ (outlier %)',
    yaxis_range=[0.35, 0.65],
    height=480, template='plotly_white',
    annotations=[dict(x=362, y=0.521, text='Paper headline', showarrow=True, arrowhead=2)]
)
fig.show()
print('\nInterpretation:', manifest['interpretation']['colab_replications'])


In [ ]:
# Cell 6 — Ramachandran scatter: ChiralFold vs wwPDB outlier %
csv_path = ROOT / 'results/ramachandran_279struct_chainfix_comparison.csv'
if csv_path.exists():
    rama = pd.read_csv(csv_path)
    rama_clean = rama[rama['wwpdb_rama_outlier_pct'] <= 50].copy()
    fig = px.scatter(
        rama_clean, x='wwpdb_rama_outlier_pct', y='chiralfold_rama_outlier_pct',
        hover_data=['pdb_id', 'bin'], opacity=0.7,
        title=f'Per-structure Ramachandran outliers (n={len(rama_clean)}, 5M2K excluded)',
        labels={'wwpdb_rama_outlier_pct': 'wwPDB outlier %', 'chiralfold_rama_outlier_pct': 'ChiralFold outlier %'},
    )
    lim = max(rama_clean['wwpdb_rama_outlier_pct'].max(), rama_clean['chiralfold_rama_outlier_pct'].max()) * 1.05
    fig.add_shape(type='line', x0=0, y0=0, x1=lim, y1=lim, line=dict(dash='dash', color='gray'))
    fig.update_layout(height=500, template='plotly_white')
    fig.show()
else:
    print('Chainfix comparison CSV not found — run from repo root.')


In [ ]:
# Cell 7 — D-residue error taxonomy
tax_rows = []
for cat, info in error_cls['classification'].items():
    tax_rows.append({'category': cat, 'structures': info['structures'], 'errors': info['errors']})
tax_df = pd.DataFrame(tax_rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Errors by category', 'Errors by CCD code'))
fig.add_trace(go.Bar(x=tax_df['category'], y=tax_df['errors'], marker_color='#6366f1', name='Errors'), row=1, col=1)
ccd = pd.Series(d_survey['errors_by_type']).sort_values(ascending=False)
fig.add_trace(go.Bar(x=ccd.index, y=ccd.values, marker_color='#f97316', name='CCD'), row=1, col=2)
fig.update_layout(title='D-residue annotation errors (29 total in 16 structures)', height=420, showlegend=False, template='plotly_white')
fig.show()

print(f"Survey: {d_survey['checkable_residues']:,} residues across {d_survey['pdb_files_scanned']:,} PDB files, {d_survey['error_rate_pct']}% error rate")
print('MolProbity does not flag any of these — L-only Cα chirality check is blind to D-residue mismatches.')


In [ ]:
# Cell 8 — Experimental validation table
rows = []
for s in exp_val['structures']:
    status = '✓ pass' if s['automated_validation_pass'] is True else ('? borderline' if s['automated_validation_pass'] is None else '✗ fail')
    rows.append({
        'PDB': s['pdb_id'],
        'Type': s['error_type'],
        'CCD': s['labeled_ccd'],
        'Method': (s.get('experimental_method') or '')[:12],
        'Res': s.get('resolution_angstrom'),
        'Status': status,
    })
val_df = pd.DataFrame(rows)
display(val_df)
print('\n5M2K benchmark exclusion:', exp_val['5m2k_exclusion']['benchmark_exclusion_reason'][:120], '...')


In [ ]:
# Cell 9 — AF3 synthetic correction benchmark
sys_rows = []
for s in af3['systems']:
    # Schema: before.n_violations + correction.violations_after (not s['after'])
    after = s.get('correction', {}).get('violations_after', s.get('after', {}).get('n_violations', 0))
    sys_rows.append({
        'System': s['id'],
        'Before violations': s['before']['n_violations'],
        'After violations': after,
        'Childs AF3 rate': f"{s['childs_af3_rate']*100:.0f}%",
    })
af3_df = pd.DataFrame(sys_rows)
display(af3_df)

fig = go.Figure(data=[
    go.Bar(name='Before correction', x=af3_df['System'], y=af3_df['Before violations'], marker_color='#ef4444'),
    go.Bar(name='After correction', x=af3_df['System'], y=af3_df['After violations'], marker_color='#22c55e'),
])
fig.update_layout(title='AF3-mimetic chirality violations (Childs et al. 2025 systems)', barmode='group', height=380, template='plotly_white')
fig.show()

print(f"Aggregate: {af3['aggregate_detection_recall_pct']:.0f}% detection, "
      f"{af3['post_correction_residual_violation_rate_pct']:.0f}% residual violations")


In [ ]:
mmcif_csv = ROOT / 'results/mmcif_d_residue_expansion.csv'
if mmcif_csv.exists():
    mdf = pd.read_csv(mmcif_csv)
    known = mmcif.get('known_error_cohort', {})
    uni = mmcif.get('universe_cohort', {})
    print(f"mmCIF scan: {mmcif.get('n_d_residues')} D-residues · "
          f"known_errors={known.get('n_errors')} · universe_errors={uni.get('n_errors')}")
    print('All error PDBs (mmCIF):', ', '.join(mmcif.get('error_pdbs', [])))
    print('Universe new errors:', uni.get('error_pdbs', []))
    assert known.get('n_errors', mmcif.get('n_errors')) == 29, 'Expected 29 known errors'
else:
    print('mmCIF expansion CSV not found')


In [ ]:
# Cell 11 — Colab demo highlights + static figures
from IPython.display import Image, display
import os

print('=== Colab Quick Demo (authoritative run) ===')
print('  PDB 1LDF audit score: 83.6/100')
print('  Chirality violations: 0')
print('  D-peptide conformers: 45/45 converged')
print('  Mirror transform: 1LDF → 1LDF_D_mirror.pdb (RMSD = 0.0 Å)\n')

print('=== Colab Toy Demo ===')
print('  AF3 synthetic correction: 1 violation → 0 after correct_af3_output\n')

static_figs = [
    'results/ramachandran_279struct_chainfix_plot.png',
    'results/colab_runs/ramachandran_100struct/ramachandran_100struct_plot.png',
    'results/af3_accuracy_comparison.png',
    'results/molprobity_comparison.png',
]
for p in static_figs:
    if os.path.isfile(p):
        print(p)
        display(Image(filename=p, width=700))

print('\n--- Interpretation summary ---')
for k, v in manifest['interpretation'].items():
    print(f'\n{k}:')
    print(' ', v)


## What these numbers mean

| Result | Takeaway |
|--------|----------|
| **D-residue survey (12,573 residues)** | ~0.23% of D-labeled residues have L coordinates. MolProbity misses all of them. |
| **Experimental validation (14/14 pass)** | Automated RCSB metadata + CCD InChI cross-check confirms errors are real annotation issues. |
| **Ramachandran ρ ≈ 0.52 (n=362)** | ChiralFold Ramachandran outlier rates rank consistently with wwPDB/MolProbity on standard proteins. |
| **Colab n=155 (ρ=0.57) / n=285 (ρ=0.44)** | Independent replications — ρ varies with sample composition; both support moderate-to-strong rank agreement. |
| **AF3 correction (100%)** | Post-processing fixes inverted stereocenters Childs et al. report at ~51% in AF3 D-peptide outputs. |
| **mmCIF re-verification (29/29)** | Native mmCIF recovers the same 29 mismatches for all 16 known-error structures. |
| **Aristotle Lean proofs** | Machine-checked in `formal/chirality_nogo/` — distance-only representations cannot recover chirality sign. |

Reproduce any benchmark: see `benchmarks/README.md` and `results/REPRODUCIBILITY.md`.  
mmCIF Colab: [`Reproduce_mmCIF_D_Residue_Survey.ipynb`](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/Reproduce_mmCIF_D_Residue_Survey.ipynb).
